# L2-only PairHead ablation, top4 data, T=10, L2 unfrozen

Actor-only ablation for the next supervised run:

```
L0 planet/fleet/comet encoders   frozen
L1 PlanetEntityEncoder           warm-started, frozen
L2 CrossEntityAttention          warm-started, trainable
L3 DualRoleAttention             removed (--skip-l34)
L4 JointRoleAttention            removed (--skip-l34)
PlayerConsolidator / value path  removed (--no-consolidator)
PairHead                         directly after L2, trainable
```

Labels remain the existing PairHead labels: `pair_logits` masked BCE on expert source-target cells plus `pair_frac` MSE on positive cells using `pair_ships / source_ships_before_launch`.

Data policy:

- Reuse `gs://orbit-wars-shipping/entity/pair_cache_top4.pt` instead of uploading a duplicate.
- Override its lazy history offsets to T=10: `(45, 40, 35, 30, 25, 20, 15, 10, 5, 0)`.
- Run a correctness audit before training.
- Drop whole episodes that contain any positive pair label whose source is not learner-owned in the current input features.


## 1. Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
print(f'GCS bucket: {BUCKET}')


## 2. Pull code, weights, warm-start, and larger pair cache

The loader pulls independent objects in parallel. If a chunk manifest exists for the selected cache it downloads chunks concurrently; otherwise it reuses the single existing object and does not create a duplicate.

In [ ]:
import concurrent.futures
import hashlib
import json
import os
import subprocess
import time
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

PAIR_CACHE_PREFIX = 'pair_cache_top4'
PAIR_CACHE = WORK / 'pair_cache.pt'
PAIR_HISTORY_OFFSETS = (45, 40, 35, 30, 25, 20, 15, 10, 5, 0)
PAIR_HISTORY_OFFSETS_ARG = ','.join(str(x) for x in PAIR_HISTORY_OFFSETS)

INIT_RUN = 'pair_T10_L3L4_withcons_d256_b128_20ep_lr0.0001_20260601-071020'
BASELINE_CKPT = WORK / 'baseline_entity_encoder_best.pt'


def run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)


def gcs_size(url: str) -> int | None:
    try:
        out = subprocess.run(
            ['gcloud', 'storage', 'objects', 'describe', url, '--format=value(size)'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError:
        return None
    s = out.stdout.strip()
    return int(s) if s else None


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


def cp_if_needed(src: str, dst: Path, *, force: bool = False):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    remote_size = gcs_size(src)
    if remote_size is None:
        raise FileNotFoundError(src)
    if dst.exists() and not force and dst.stat().st_size == remote_size:
        print(f'  cached {dst.name} ({remote_size/1024**3:.2f} GiB)')
        return dst.name, 0.0, remote_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} -> {dst.name} ...', flush=True)
    run(['gcloud', 'storage', 'cp', src, str(dst)])
    return dst.name, time.time() - t0, dst.stat().st_size


def pull_chunked_or_single(bucket: str, object_name: str, dst: Path):
    dst = Path(dst)
    manifest_local = WORK / f'{Path(object_name).stem}.manifest.json'
    manifest = None
    manifest_src = None
    for cand in (f'{object_name}.manifest.json', f'{Path(object_name).stem}.manifest.json'):
        src = f'{bucket}/{cand}'
        if gcs_size(src) is None:
            continue
        cp_if_needed(src, manifest_local, force=True)
        manifest = json.loads(manifest_local.read_text())
        manifest_src = src
        break
    if manifest is None:
        return cp_if_needed(f'{bucket}/{object_name}', dst)

    chunks = manifest.get('chunks') or manifest.get('parts') or []
    total_bytes = int(manifest.get('total_bytes') or manifest.get('bytes') or 0)
    if dst.exists() and total_bytes and dst.stat().st_size == total_bytes:
        print(f'  cached assembled {dst.name} from {manifest_src}')
        return dst.name, 0.0, dst.stat().st_size
    if not chunks:
        return cp_if_needed(f'{bucket}/{object_name}', dst)

    part_dir = WORK / f'{Path(object_name).stem}_parts'
    part_dir.mkdir(parents=True, exist_ok=True)
    max_workers = max(1, min(16, len(chunks)))
    print(f'  pulling {len(chunks)} chunks for {object_name} with {max_workers} workers ...', flush=True)

    def pull_part(spec):
        name = spec['name']
        src = name if str(name).startswith('gs://') else f'{bucket}/{name}'
        part = part_dir / Path(name).name
        expected = int(spec.get('size_bytes') or spec.get('bytes') or spec.get('size') or 0)
        expected_sha = spec.get('sha256')
        if part.exists() and (not expected or part.stat().st_size == expected):
            if expected_sha is None or sha256_file(part) == expected_sha:
                return part
        if part.exists():
            part.unlink()
        run(['gcloud', 'storage', 'cp', src, str(part)])
        if expected and part.stat().st_size != expected:
            raise RuntimeError(f'chunk size mismatch for {name}: {part.stat().st_size} != {expected}')
        if expected_sha and sha256_file(part) != expected_sha:
            raise RuntimeError(f'chunk sha mismatch for {name}')
        return part

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        list(pool.map(pull_part, chunks))

    if dst.exists():
        dst.unlink()
    print(f'  assembling -> {dst.name} ...', flush=True)
    with dst.open('wb') as out:
        for spec in chunks:
            part = part_dir / Path(spec['name']).name
            with part.open('rb') as fh:
                for block in iter(lambda: fh.read(8 << 20), b''):
                    out.write(block)
    if total_bytes and dst.stat().st_size != total_bytes:
        raise RuntimeError(f'assembled size mismatch for {dst}: {dst.stat().st_size} != {total_bytes}')
    return dst.name, 0.0, dst.stat().st_size

small_tasks = [
    (f'{BUCKET}/code.tgz', WORK / 'code.tgz'),
    (f'{BUCKET}/weights.tgz', WORK / 'weights.tgz'),
    (f'{BUCKET}/runs/{INIT_RUN}/entity_encoder_best.pt', BASELINE_CKPT),
]

started = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(small_tasks) + 1) as pool:
    futures = [pool.submit(cp_if_needed, src, dst) for src, dst in small_tasks]
    futures.append(pool.submit(pull_chunked_or_single, BUCKET, f'{PAIR_CACHE_PREFIX}.pt', PAIR_CACHE))
    for fut in concurrent.futures.as_completed(futures):
        results.append(fut.result())

for name, dt, size in sorted(results, key=lambda row: -row[2]):
    print(f'  {name:<36s} {size/1024**2:>10.1f} MiB  {dt:7.1f}s')
print(f'total wall: {time.time()-started:.1f}s')


## 3. Extract code and verify architecture flags

In [ ]:
# Keep the large cache and warm-start checkpoint; refresh Python code and L0 weights.
!rm -rf agents scripts ckpts
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' ! -name 'baseline_entity_encoder_best.pt' -delete
!tar xzf code.tgz
!tar xzf weights.tgz

import gc
import importlib
import sys
for name in list(sys.modules):
    if name.startswith('agents') or name.startswith('scripts'):
        del sys.modules[name]
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true


In [ ]:
import inspect
import torch
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel, train

sig = inspect.signature(train)
assert 'pair_history_offsets' in sig.parameters, 'stale code.tgz: missing pair_history_offsets'
assert 'drop_invalid_source_episodes' in sig.parameters, 'stale code.tgz: missing invalid-source filter'

m = EntityPretrainModel(
    d_model=256, n_steps=len(PAIR_HISTORY_OFFSETS), d_pair=256,
    conditioner_n_layers=3, head_n_layers=3,
    skip_l34=True, with_consolidator=False,
)
assert m.dual_role is None and m.joint_role is None, 'skip_l34 did not remove L3/L4'
assert m.consolidator is None, '--no-consolidator path should remove PlayerConsolidator'
print(f'architecture OK: L3={m.dual_role}, L4={m.joint_role}, consolidator={m.consolidator}')
print(f'params: total={sum(p.numel() for p in m.parameters()):,}')
del m


## 4. Dataset correctness audit

This checks the labels before training. The top4 cache has a small number of source-owner-invalid positives in the current local audit, so training uses `--drop-invalid-source-episodes` below.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import torch
from scripts.build_pair_dataset_orbital_occle import CachedPairDataset
from agents.transformer_v2.featurizer.fleet_featurizer import SHIPS_LOG_MAX

EXPECTED_OFFSETS = tuple(PAIR_HISTORY_OFFSETS)
ds = CachedPairDataset(PAIR_CACHE)
ds.history_offsets = EXPECTED_OFFSETS
ds.config = dict(ds.config)
ds.config['history_offsets'] = list(EXPECTED_OFFSETS)

compact_config = {k: v for k, v in ds.config.items() if k != 'snapshot_to_player'}
print('compact config:', compact_config)
assert tuple(ds.history_offsets) == EXPECTED_OFFSETS
assert int(ds.config.get('max_planets')) == 64
assert int(ds.config.get('max_fleets')) == 1024

acted_indices = list(getattr(ds, 'acted_indices', [])) or [i for i, s in enumerate(ds.snapshots) if bool(s['pair_labels'].any())]
train_indices = list(range(len(ds))) if ds.config.get('keep_non_acted') else acted_indices
print(f'rows: context={len(ds):,} acted={len(acted_indices):,} supervised_before_filter={len(train_indices):,}')

keyset = set(ds.keys)
hist_counts = Counter()
for i in train_indices:
    ep, t = ds.keys[i]
    have = sum(1 for off in EXPECTED_OFFSETS if (ep, t - off) in keyset)
    hist_counts[have] += 1
print(f'T=10 history counts: {dict(sorted(hist_counts.items()))}')
print(f'T=10 complete={hist_counts[len(EXPECTED_OFFSETS)]/max(1,len(train_indices)):.3f}')

errs = Counter()
by_player = defaultdict(Counter)
bad_source_episodes = set()
snapshot_to_player = ds.config.get('snapshot_to_player') or []
valid_cells = 0
pos_cells = 0
ship_sum = 0
max_frac = 0.0

for idx, snap in enumerate(ds.snapshots):
    pm = snap['planet_mask'].bool()
    pl = snap['pair_labels'].bool()
    pv = snap['pair_valid'].bool()
    ps = snap.get('pair_ships')
    expected_valid = pm[:, None] & pm[None, :]
    expected_valid.fill_diagonal_(False)
    if not torch.equal(pv, expected_valid):
        errs['pair_valid_mismatch'] += 1
    if bool((pl & ~pv).any()):
        errs['positive_outside_pair_valid'] += 1
    if bool(torch.diag(pl).any()):
        errs['diagonal_positive'] += 1
    if ps is not None:
        if not torch.equal(ps > 0, pl):
            errs['pair_ships_label_mismatch'] += 1
        ship_sum += int(ps.sum().item())
    p = int(pl.sum().item())
    v = int(pv.sum().item())
    pos_cells += p
    valid_cells += v
    if p:
        player = snapshot_to_player[idx] if idx < len(snapshot_to_player) else '?'
        nz = torch.nonzero(pl, as_tuple=False)
        src = nz[:, 0]
        tgt = nz[:, 1]
        owner0 = snap['planet_features'][src, 1]
        bad = owner0 < 0.5
        by_player[player]['pos'] += int(nz.shape[0])
        by_player[player]['bad_src_owner'] += int(bad.sum().item())
        if bool(bad.any()):
            bad_source_episodes.add(str(ds.keys[idx][0]))
        if ps is not None:
            src_ships = torch.expm1(snap['planet_features'][src, 6].clamp(min=0) * SHIPS_LOG_MAX)
            sent = ps[src, tgt].float()
            frac = (sent / src_ships.clamp(min=1)).clamp(min=0)
            if frac.numel():
                max_frac = max(max_frac, float(frac.max().item()))
            if bool((sent - src_ships > 1e-3).any()):
                errs['sent_gt_source_ships'] += int((sent - src_ships > 1e-3).sum().item())

print(f'labels: positives={pos_cells:,} valid_cells={valid_cells:,} pos_frac={pos_cells/max(1, valid_cells):.6f} total_ships={ship_sum:,} max_sent_frac={max_frac:.3f}')
print('bad source owner by player:')
for player, ctr in sorted(by_player.items()):
    print(f'  {player}: {dict(ctr)}')
print(f'bad source episodes to drop: {len(bad_source_episodes):,}')
print('errors:', dict(errs) if errs else '{}')

hard_errors = {k: v for k, v in errs.items() if k not in {'sent_gt_source_ships'}}
assert not hard_errors, hard_errors
assert pos_cells > 0

# Verify lazy T=10 stacking leaves labels current-turn-only.
for i in train_indices[:64]:
    item = ds[i]
    assert tuple(item['planet_features'].shape[:2]) == (len(EXPECTED_OFFSETS), 64)
    assert item['pair_labels'].dim() == 2 and item['pair_valid'].dim() == 2
    assert torch.equal(item['pair_labels'], ds.snapshots[i]['pair_labels'])
    assert torch.equal(item['pair_valid'], ds.snapshots[i]['pair_valid'])
print('dataset correctness audit complete')


## 5. Stage cache path and checkpoints

In [ ]:
import os
import shutil
from pathlib import Path

CACHE_DIR = Path('data/datasets/_pair_cache/bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TARGET = CACHE_DIR / 'bowwowforeach_Ebi_ErfanEshratifar_Shun_PI_T6_p64_f1024_all.pt'
if CACHE_TARGET.exists():
    CACHE_TARGET.unlink()
os.link(PAIR_CACHE, CACHE_TARGET)
PAIR_CACHE_PATH = str(CACHE_TARGET)
print(f'cache staged: {PAIR_CACHE_PATH} ({CACHE_TARGET.stat().st_size/1024**3:.2f} GiB)')

PLANET_RUN_DIR = WORK / 'ckpts/planet'
FLEET_RUN_DIR = WORK / 'ckpts/fleet'
COMET_RUN_DIR = WORK / 'ckpts/comet'
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy(WORK / 'fleet_encoder_best.pt', FLEET_RUN_DIR / 'fleet_encoder_best.pt')
shutil.copy(WORK / 'comet_past_best.pt', COMET_RUN_DIR / 'comet_past_best.pt')

INIT_FROM_ENTITY = str(BASELINE_CKPT)
for tag, path in (
    ('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
    ('fleet', FLEET_RUN_DIR / 'fleet_encoder_best.pt'),
    ('comet', COMET_RUN_DIR / 'comet_past_best.pt'),
    ('actor_init', Path(INIT_FROM_ENTITY)),
):
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    print(f'{tag:10s}: d_model={cfg.get("d_model")} n_steps={cfg.get("n_steps")} epoch={ckpt.get("epoch")}')


## 6. Train

Key flags:

- `--skip-l34`: PairHead directly after L2.
- `--no-consolidator`: no PlayerConsolidator/value branch in the saved actor checkpoint.
- `--freeze-l1-only`: L1 frozen; L2 and PairHead trainable.
- `--pair-history-offsets`: use T=10 from the existing single-frame cache.
- `--drop-invalid-source-episodes`: remove suspect mixed-cache episodes before split.

In [ ]:
D_MODEL = 256
D_PAIR = 256
ENTITY_N_HEADS = 8
CROSS_N_HEADS = 8
CROSS_N_LAYERS = 2
DUAL_N_HEADS = 8
CONDITIONER_N_LAYERS = 3
HEAD_N_LAYERS = 3
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 20
LR = 2e-5
WEIGHT_DECAY = 1e-4
SEED = 1729
MAX_PLANETS = 64
MAX_FLEETS = 1024
PAIR_POS_WEIGHT = 600.0
VAL_FRAC = 0.10
TEST_FRAC = 0.10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'L2only_noL34_novalue_top4T10_dropbad_d{D_MODEL}_b{BATCH_SIZE}_{EPOCHS}ep_lr{LR:g}_{TS}'
OUT_DIR = f'data/runs/entity/{RUN_TAG}'
print('out dir:', OUT_DIR)
print('init:', INIT_FROM_ENTITY)
print('history offsets:', PAIR_HISTORY_OFFSETS_ARG)
print('device:', DEVICE)


In [ ]:
!python -u -m agents.transformer_v2.pretrain.entity_encoder   --planet-run-dir $PLANET_RUN_DIR   --fleet-run-dir  $FLEET_RUN_DIR   --comet-run-dir  $COMET_RUN_DIR   --pair-cache-path $PAIR_CACHE_PATH   --pair-history-offsets $PAIR_HISTORY_OFFSETS_ARG   --drop-invalid-source-episodes   --out-dir $OUT_DIR   --d-model $D_MODEL   --d-pair $D_PAIR   --entity-n-heads $ENTITY_N_HEADS   --cross-n-heads $CROSS_N_HEADS   --cross-n-layers $CROSS_N_LAYERS   --dual-n-heads $DUAL_N_HEADS   --conditioner-n-layers $CONDITIONER_N_LAYERS   --head-n-layers $HEAD_N_LAYERS   --skip-l34   --no-consolidator   --freeze-l1-only   --init-from-entity-ckpt $INIT_FROM_ENTITY   --batch-size $BATCH_SIZE   --epochs $EPOCHS   --lr $LR   --weight-decay $WEIGHT_DECAY   --max-planets $MAX_PLANETS   --max-fleets $MAX_FLEETS   --pair-pos-weight $PAIR_POS_WEIGHT   --val-frac $VAL_FRAC   --test-frac $TEST_FRAC   --num-workers $NUM_WORKERS   --seed $SEED   --device $DEVICE


## 7. Upload run

In [ ]:
src = Path(OUT_DIR)
assert src.is_dir(), src
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent], check=True)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{dst_parent}{src.name}/'], check=False)
